# CS452 Assignment 1: Affective Computing on Facial Expressions

**Clean, optimized version for CPU training**

## Overview
- **Goal**: Facial Expression classification (8 classes) + Valence/Arousal regression ([-1, 1])
- **Dataset**: 3999 face images with .npy annotations
- **Models**: ResNet50, MobileNetV2 with multi-task heads
- **Evaluation**: Accuracy, F1, Kappa, AUC; RMSE, CORR, SAGR, CCC


In [1]:
# 1) Setup & Imports
import os, time, random, math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
from torchvision import transforms
from torchvision.models import resnet50, mobilenet_v2
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, roc_auc_score
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from PIL import Image
from dataclasses import dataclass

# Set seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cpu')
print('Device:', device)


Device: cpu


c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 2) Configuration
@dataclass
class Config:
    data_root: str = "DL_Assignment1_Dataset/Dataset/Dataset"  # Correct path
    img_size: int = 160
    batch_size: int = 12
    epochs: int = 15
    num_classes: int = 8
    lr: float = 3e-4
    val_split: float = 0.1
    max_train_batches: int = 100
    max_val_batches: int = 30
    run_resnet: bool = True
    run_mobilenet: bool = True
    out_dir: str = "outputs"

CFG = Config()
os.makedirs(CFG.out_dir, exist_ok=True)
print(f"Config: {CFG.img_size}x{CFG.img_size}, {CFG.epochs} epochs, batch={CFG.batch_size}")
print(f"Data path: {CFG.data_root}")


Config: 160x160, 15 epochs, batch=12


In [3]:
# 3) Dataset
class AffectDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform
        self.samples = self._load_samples()
    
    def _load_samples(self):
        samples = []
        images_dir = os.path.join(self.root, "images")
        ann_dir = os.path.join(self.root, "annotations")
        
        for fname in os.listdir(images_dir):
            if not fname.endswith('.jpg'):
                continue
            
            sample_id = fname[:-4]
            exp_path = os.path.join(ann_dir, f"{sample_id}_exp.npy")
            val_path = os.path.join(ann_dir, f"{sample_id}_val.npy")
            aro_path = os.path.join(ann_dir, f"{sample_id}_aro.npy")
            
            if not all(os.path.exists(p) for p in [exp_path, val_path, aro_path]):
                continue
            
            try:
                exp = int(np.load(exp_path))
                val = float(np.load(val_path))
                aro = float(np.load(aro_path))
                
                if val == -2 or aro == -2:  # Skip uncertain
                    continue
                
                samples.append({
                    'img': os.path.join(images_dir, fname),
                    'exp': exp,
                    'val': val,
                    'aro': aro
                })
            except:
                continue
        
        return samples
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        item = self.samples[idx]
        image = Image.open(item['img']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, item['exp'], np.array([item['val'], item['aro']], dtype=np.float32)

# Transforms
train_tfms = transforms.Compose([
    transforms.Resize((CFG.img_size, CFG.img_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_tfms = transforms.Compose([
    transforms.Resize((CFG.img_size, CFG.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
dataset = AffectDataset(CFG.data_root, transform=train_tfms)
print(f'Loaded {len(dataset)} samples')

# Split
val_size = int(len(dataset) * CFG.val_split)
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])
val_ds.dataset.transform = val_tfms

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=0)
print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'Dataset/Dataset\\images'

In [ ]:
# 4) Metrics
def compute_classification_metrics(y_true, y_pred_logits):
    y_true = np.asarray(y_true)
    y_pred_logits = np.asarray(y_pred_logits)
    y_pred = y_pred_logits.argmax(axis=1)
    
    acc = float(accuracy_score(y_true, y_pred))
    f1 = float(f1_score(y_true, y_pred, average='weighted'))
    kappa = float(cohen_kappa_score(y_true, y_pred))
    
    try:
        prob = torch.softmax(torch.tensor(y_pred_logits), dim=1).numpy()
        roc_auc = float(roc_auc_score(y_true, prob, multi_class='ovr'))
    except:
        roc_auc = float('nan')
    
    return {'accuracy': acc, 'f1': f1, 'kappa': kappa, 'roc_auc': roc_auc}

def compute_continuous_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    corr = float(np.corrcoef(y_true.flatten(), y_pred.flatten())[0, 1]) if len(y_true) > 1 else 0.0
    sagr = float(np.mean(np.sign(y_true) == np.sign(y_pred)))
    
    return {'rmse': rmse, 'corr': corr, 'sagr': sagr}


In [ ]:
# 5) Models
class MultiTaskHead(nn.Module):
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(in_features, in_features // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(in_features // 2, num_classes)
        )
        self.regressor = nn.Sequential(
            nn.Linear(in_features, in_features // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(in_features // 2, 2),
            nn.Tanh()  # [-1, 1]
        )
    
    def forward(self, x):
        return self.classifier(x), self.regressor(x)

def build_resnet50(num_classes):
    base = resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
    in_features = base.fc.in_features
    base.fc = nn.Identity()
    head = MultiTaskHead(in_features, num_classes)
    return nn.Sequential(base, nn.Flatten(), head)

def build_mobilenet_v2(num_classes):
    base = mobilenet_v2(weights=torchvision.models.MobileNet_V2_Weights.IMAGENET1K_V2)
    in_features = base.classifier[-1].in_features
    base.classifier = nn.Sequential(nn.Dropout(0.2), nn.Identity())
    head = MultiTaskHead(in_features, num_classes)
    return nn.Sequential(base, nn.Flatten(), head)

class Criterion(nn.Module):
    def __init__(self, cls_weight=1.0, reg_weight=1.0):
        super().__init__()
        self.cls_loss = nn.CrossEntropyLoss()
        self.reg_loss = nn.SmoothL1Loss()
        self.cls_weight = cls_weight
        self.reg_weight = reg_weight
    
    def forward(self, outputs, targets):
        logits, va_pred = outputs
        y_cls, y_va = targets
        loss_cls = self.cls_loss(logits, y_cls)
        loss_reg = self.reg_loss(va_pred, y_va)
        return self.cls_weight * loss_cls + self.reg_weight * loss_reg


In [ ]:
# 6) Training
class Trainer:
    def __init__(self, model, optimizer, criterion):
        self.model = model
        self.optimizer = optimizer
        self.criterion = criterion
    
    def train_epoch(self, loader):
        self.model.train()
        losses = []
        pbar = tqdm(loader, desc="Training", leave=False)
        
        for step, (images, y_cls, y_va) in enumerate(pbar):
            if step >= CFG.max_train_batches:
                break
            
            images = images.to(device)
            y_cls = y_cls.to(device)
            y_va = y_va.to(device)
            
            self.optimizer.zero_grad()
            logits, va_pred = self.model(images)
            loss = self.criterion((logits, va_pred), (y_cls, y_va))
            loss.backward()
            self.optimizer.step()
            
            losses.append(loss.item())
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        return float(np.mean(losses))
    
    def evaluate(self, loader):
        self.model.eval()
        all_logits, all_va_pred, all_cls, all_va = [], [], [], []
        pbar = tqdm(loader, desc="Evaluating", leave=False)
        
        with torch.no_grad():
            for step, (images, y_cls, y_va) in enumerate(pbar):
                if step >= CFG.max_val_batches:
                    break
                
                images = images.to(device)
                logits, va_pred = self.model(images)
                
                all_logits.append(logits.cpu().numpy())
                all_va_pred.append(va_pred.cpu().numpy())
                all_cls.append(y_cls.numpy())
                all_va.append(y_va.numpy())
                
                pbar.set_postfix({'samples': len(all_cls) * CFG.batch_size})
        
        logits = np.concatenate(all_logits, axis=0) if all_logits else np.zeros((0, CFG.num_classes))
        y_cls = np.concatenate(all_cls, axis=0) if all_cls else np.zeros((0,), dtype=int)
        y_va_pred = np.concatenate(all_va_pred, axis=0) if all_va_pred else np.zeros((0, 2))
        y_va = np.concatenate(all_va, axis=0) if all_va else np.zeros((0, 2))
        
        cls_metrics = compute_classification_metrics(y_cls, logits)
        cont_metrics = compute_continuous_metrics(y_va, y_va_pred)
        return cls_metrics, cont_metrics
    
    def fit(self, train_loader, val_loader, epochs):
        history = {'train_loss': [], 'val_cls': [], 'val_cont': []}
        
        for epoch in range(1, epochs + 1):
            print(f"\nEpoch {epoch}/{epochs}")
            
            t0 = time.time()
            train_loss = self.train_epoch(train_loader)
            val_cls, val_cont = self.evaluate(val_loader)
            t1 = time.time()
            
            history['train_loss'].append(train_loss)
            history['val_cls'].append(val_cls)
            history['val_cont'].append(val_cont)
            
            print(f"Train Loss: {train_loss:.4f}")
            print(f"Val Acc: {val_cls['accuracy']:.4f}, F1: {val_cls['f1']:.4f}")
            print(f"Val RMSE: {val_cont['rmse']:.4f}, Corr: {val_cont['corr']:.4f}")
            print(f"Time: {t1-t0:.1f}s")
        
        return history


In [ ]:
# 7) Run Experiments
def run_experiment(model_name):
    print(f"\n{'='*50}")
    print(f"Training {model_name.upper()}")
    print(f"{'='*50}")
    
    # Build model
    if model_name.lower() == 'resnet50':
        model = build_resnet50(CFG.num_classes)
    elif model_name.lower() == 'mobilenetv2':
        model = build_mobilenet_v2(CFG.num_classes)
    else:
        raise ValueError('Unknown model')
    
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=1e-4)
    criterion = Criterion()
    trainer = Trainer(model, optimizer, criterion)
    
    # Train
    history = trainer.fit(train_loader, val_loader, CFG.epochs)
    
    # Save
    torch.save(model.state_dict(), os.path.join(CFG.out_dir, f"{model_name}_weights.pth"))
    
    return history

# Run experiments
hists = {}
if CFG.run_resnet:
    hists['resnet50'] = run_experiment('resnet50')
if CFG.run_mobilenet:
    hists['mobilenetv2'] = run_experiment('mobilenetv2')


In [ ]:
# 8) Results & Visualization
# Plot training curves
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
for name, hist in hists.items():
    plt.plot(hist['train_loss'], label=name, marker='o')
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
for name, hist in hists.items():
    accs = [h['accuracy'] for h in hist['val_cls']]
    plt.plot(accs, label=name, marker='o')
plt.title('Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Final results
print(f"\n{'='*60}")
print("FINAL RESULTS")
print(f"{'='*60}")

for name, hist in hists.items():
    print(f"\n{name.upper()}:")
    print(f"  Classification: {hist['val_cls'][-1]}")
    print(f"  Continuous: {hist['val_cont'][-1]}")

print(f"\n{'='*60}")
print("Training completed successfully!")
print(f"Results saved in: {CFG.out_dir}")
